In [6]:
import carla
from carla import Transform, Location, Rotation 
import random
import sys

sys.path.append("./CARLA_0.9.16") # Add the folder to the path

from PythonAPI.carla.agents.navigation.basic_agent import BasicAgent
from PythonAPI.carla.agents.navigation.behavior_agent import BehaviorAgent


client = carla.Client('localhost', 2000)
client.set_timeout(5.0)
world = client.get_world()
transform = Transform(Location(x=50, y=27, z=1), Rotation(yaw=180))
spawn_points = world.get_map().get_spawn_points()

bp_lib = world.get_blueprint_library()

# Option A: exact id (b
bp = bp_lib.find('vehicle.micro.microlino')

actor = world.try_spawn_actor(bp, transform)
print("spawned:", actor)

if actor is None:
    raise RuntimeError("Vehicle failed to spawn")

agent = BehaviorAgent(actor, behavior='aggressive')   # or BasicAgent(actor)

t = actor.get_transform()
behind = t.location - t.get_forward_vector() * 8 + carla.Location(z=3)


spectator = world.get_spectator()
spectator.set_transform(carla.Transform(
    behind,
    carla.Rotation(pitch=-10, yaw=t.rotation.yaw)
))

spawned: Actor(id=28, type=vehicle.micro.microlino)


In [7]:
destination = random.choice(spawn_points).location
agent.set_destination(destination)


In [8]:
while True:
    if agent.done():
        print("The target has been reached, stopping the simulation")
        break
    actor.apply_control(agent.run_step())

    t = actor.get_transform()
    # behind + a little up
    behind = t.location - t.get_forward_vector() * 8 + carla.Location(z=3)

    spectator.set_transform(carla.Transform(
        behind,
        carla.Rotation(pitch=-10, yaw=t.rotation.yaw)
    ))
    

The target has been reached, stopping the simulation


In [9]:
actor.destroy()

True